In [ ]:
!nvidia-smi

Tue Apr 28 03:25:58 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   34C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
import os, random, glob
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, models
from transformers import ViTModel
from PIL import Image
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    precision_score, recall_score, f1_score,
    confusion_matrix, accuracy_score, roc_curve
)
from torch import amp
import json, matplotlib.pyplot as plt

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

from google.colab import drive
drive.mount('/content/drive')


Device: cuda
Mounted at /content/drive


In [ ]:
TRAIN_ROOT  = "/content/drive/MyDrive/DrishtiGS/Training/Images"
TEST_ROOT   = "/content/drive/MyDrive/DrishtiGS/Test/Images"
SAVE_DIR    = "/content/drive/MyDrive/DrishtiGS/models"
METRICS_DIR = "/content/drive/MyDrive/REFUGE2_RESULTS"
os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(METRICS_DIR, exist_ok=True)

CLASS_MAP = {"normal":0, "NORMAL":0, "Normal":0,
             "glaucoma":1, "GLAUCOMA":1, "Glaucoma":1}

def collect(root):
    recs = []
    for cls in os.listdir(root):
        if cls not in CLASS_MAP: continue
        for ext in ("*.jpg","*.png","*.jpeg","*.JPG","*.PNG"):
            for p in glob.glob(os.path.join(root, cls, ext)):
                recs.append([p, CLASS_MAP[cls]])
    return pd.DataFrame(recs, columns=["image","label"])

train_df = collect(TRAIN_ROOT)
test_df  = collect(TEST_ROOT)

print(f"Train: {len(train_df)}")
print(train_df["label"].value_counts().to_string())
print(f"\nTest : {len(test_df)}")
print(test_df["label"].value_counts().to_string())


Train: 50
label
1    32
0    18

Test : 51
label
1    38
0    13


In [ ]:
IMG_SIZE = 224

train_tfm = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])
test_tfm = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

In [ ]:
class FundusDataset(Dataset):
    def __init__(self, df, transform):
        self.df = df.reset_index(drop=True)
        self.transform = transform
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        img   = Image.open(self.df.loc[idx,"image"]).convert("RGB")
        img   = self.transform(img)
        label = torch.tensor(self.df.loc[idx,"label"], dtype=torch.float32)
        return img, label

train_ds = FundusDataset(train_df, train_tfm)
test_ds  = FundusDataset(test_df,  test_tfm)

# Weighted sampler for class imbalance
labels_arr    = train_df["label"].values
class_counts  = np.bincount(labels_arr)
class_weights = 1.0 / class_counts
sample_w      = class_weights[labels_arr]
sampler       = WeightedRandomSampler(sample_w, len(sample_w))

train_loader = DataLoader(train_ds, batch_size=8, sampler=sampler, num_workers=2)
test_loader  = DataLoader(test_ds,  batch_size=8, shuffle=False,  num_workers=2)


In [ ]:
class HybridResNetViT(nn.Module):
    def __init__(self, dropout=0.3):
        super().__init__()
        self.cnn = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
        self.cnn.fc = nn.Identity()                      # → 2048-d
        self.vit = ViTModel.from_pretrained("google/vit-base-patch16-224-in21k")
        # Use pooler_output for classification (768-d)
        self.classifier = nn.Sequential(
            nn.Linear(2048 + 768, 512),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, 1)
        )

    def forward(self, x):
        cnn_feat = self.cnn(x)                            # (B, 2048)
        vit_feat = self.vit(pixel_values=x).pooler_output # (B, 768)
        return self.classifier(torch.cat([cnn_feat, vit_feat], dim=1))

model     = HybridResNetViT().to(DEVICE)
pos_weight = torch.tensor([class_counts[0]/class_counts[1]], dtype=torch.float32).to(DEVICE)
criterion  = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer  = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler  = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=15, eta_min=1e-6)
scaler     = amp.GradScaler(enabled=torch.cuda.is_available())

print(f"Model parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")


Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 191MB/s]
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/502 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

Model parameters: 111,471,169


In [ ]:
EPOCHS    = 15
best_auc  = 0.0
MODEL_PT  = os.path.join(SAVE_DIR, "hybrid_resnet_vit_best_v2.pth")

auc_hist, loss_hist = [], []

for epoch in range(1, EPOCHS + 1):
    # Train
    model.train()
    run_loss = 0.0
    for imgs, labels in train_loader:
        imgs   = imgs.to(DEVICE)
        labels = labels.unsqueeze(1).to(DEVICE)
        optimizer.zero_grad()
        with amp.autocast(device_type="cuda", enabled=torch.cuda.is_available()):
            logits = model(imgs)
            loss   = criterion(logits, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer); scaler.update()
        run_loss += loss.item() * len(imgs)
    train_loss = run_loss / len(train_loader.dataset)

    # Evaluate
    model.eval()
    y_true_ep, y_prob_ep = [], []
    with torch.no_grad():
        for imgs, labels in test_loader:
            imgs  = imgs.to(DEVICE)
            probs = torch.sigmoid(model(imgs).squeeze(1)).cpu().numpy()
            y_prob_ep.extend(probs.tolist())
            y_true_ep.extend(labels.numpy().tolist())

    val_auc = roc_auc_score(y_true_ep, y_prob_ep)
    auc_hist.append(val_auc); loss_hist.append(train_loss)

    if val_auc > best_auc:
        best_auc = val_auc
        torch.save(model.state_dict(), MODEL_PT)
        tag = "  saved"
    else:
        tag = ""

    scheduler.step()
    print(f"Ep {epoch:02d}/{EPOCHS} | Loss={train_loss:.4f} | "
          f"AUC={val_auc:.4f}{tag}")

print(f"\nBest Test AUC: {best_auc:.4f}")

Ep 01/15 | Loss=0.5344 | AUC=0.5587  saved
Ep 02/15 | Loss=0.4921 | AUC=0.6255  saved
Ep 03/15 | Loss=0.4716 | AUC=0.7328  saved
Ep 04/15 | Loss=0.4072 | AUC=0.7186
Ep 05/15 | Loss=0.4417 | AUC=0.7368  saved
Ep 06/15 | Loss=0.3673 | AUC=0.7611  saved
Ep 07/15 | Loss=0.3777 | AUC=0.7024
Ep 08/15 | Loss=0.2674 | AUC=0.6802
Ep 09/15 | Loss=0.3281 | AUC=0.7429
Ep 10/15 | Loss=0.3068 | AUC=0.7571
Ep 11/15 | Loss=0.2770 | AUC=0.7794  saved
Ep 12/15 | Loss=0.3057 | AUC=0.7713
Ep 13/15 | Loss=0.3017 | AUC=0.7834  saved
Ep 14/15 | Loss=0.3118 | AUC=0.7814
Ep 15/15 | Loss=0.2682 | AUC=0.7713

Best Test AUC: 0.7834


In [ ]:
model.load_state_dict(torch.load(MODEL_PT, map_location=DEVICE))
model.eval()

y_true_all, y_prob_all = [], []
with torch.no_grad():
    for imgs, labels in test_loader:
        imgs  = imgs.to(DEVICE)
        probs = torch.sigmoid(model(imgs).squeeze(1)).cpu().numpy()
        y_prob_all.extend(probs.tolist())
        y_true_all.extend(labels.numpy().tolist())

y_true = np.array(y_true_all)
y_prob = np.array(y_prob_all)

fpr_v, tpr_v, thresholds = roc_curve(y_true, y_prob)
best_thresh = float(thresholds[np.argmax(tpr_v - fpr_v)])
y_pred      = (y_prob >= best_thresh).astype(int)

tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

print("=" * 57)
print("  NB07 — Hybrid ResNet50+ViT | Drishti-GS Test (n=51)")
print("=" * 57)
print(f"  Threshold (Youden): {best_thresh:.4f}")
print(f"  Accuracy          : {accuracy_score(y_true, y_pred):.4f}")
print(f"  Precision         : {precision_score(y_true, y_pred, zero_division=0):.4f}")
print(f"  Sensitivity       : {recall_score(y_true, y_pred, zero_division=0):.4f}")
print(f"  Specificity       : {tn / (tn + fp):.4f}")
print(f"  F1-Score          : {f1_score(y_true, y_pred, zero_division=0):.4f}")
print(f"  ROC-AUC           : {roc_auc_score(y_true, y_prob):.4f}")
print(f"  PR-AUC            : {average_precision_score(y_true, y_prob):.4f}")
print(f"  TP={tp}  TN={tn}  FP={fp}  FN={fn}")
print("=" * 57)

metrics_nb07 = {
    "model": "Hybrid ResNet50+ViT",
    "dataset": "Drishti-GS",
    "n_test": int(len(y_true)),
    "threshold": round(float(best_thresh), 4),
    "accuracy": round(float(accuracy_score(y_true, y_pred)), 4),
    "precision": round(float(precision_score(y_true, y_pred, zero_division=0)), 4),
    "sensitivity": round(float(recall_score(y_true, y_pred, zero_division=0)), 4),
    "specificity": round(float(tn / (tn + fp)), 4),
    "f1": round(float(f1_score(y_true, y_pred, zero_division=0)), 4),
    "roc_auc": round(float(roc_auc_score(y_true, y_prob)), 4),
    "pr_auc": round(float(average_precision_score(y_true, y_prob)), 4),
    "TP": int(tp), "TN": int(tn), "FP": int(fp), "FN": int(fn)
}
with open(f"{METRICS_DIR}/nb07_hybrid_drishti_metrics.json", "w") as f:
    json.dump(metrics_nb07, f, indent=2)
print(f"Saved: {METRICS_DIR}/nb07_hybrid_drishti_metrics.json")



  NB07 — Hybrid ResNet50+ViT | Drishti-GS Test (n=51)
  Threshold (Youden): 0.5564
  Accuracy          : 0.7451
  Precision         : 0.9032
  Sensitivity       : 0.7368
  Specificity       : 0.7692
  F1-Score          : 0.8116
  ROC-AUC           : 0.7834
  PR-AUC            : 0.9102
  TP=28  TN=10  FP=3  FN=10
Saved: /content/drive/MyDrive/REFUGE2_RESULTS/nb07_hybrid_drishti_metrics.json
